# cAIuldron - AI Recipe Generator Pipeline (v3 - Optimized)

**Complete pipeline: Photo → Ingredient Detection → Nutrition → Recipes**

## Features:
- 🔍 Multi-ingredient detection (CLIP + DETR)
- 🥗 Nutrition estimation (529 ingredients)
- 🍳 Multi-model recipe generation (GPT-2, Llama 3.2 1B, Llama 3.1 8B)
- 🔧 Auto-fix recipe formatting & titles
- 🌐 Beautiful Gradio web interface
- 💯 100% local, no API costs

In [1]:
from llama_cpp import Llama

## 1. Environment Setup

In [2]:
import os
import sys
import json
import time
import re
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

from transformers import CLIPProcessor, CLIPModel, DetrImageProcessor, DetrForObjectDetection
import torch

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Packages imported successfully")
print(f"  - PyTorch version: {torch.__version__}")
print(f"  - CUDA available: {torch.cuda.is_available()}")

✓ Packages imported successfully
  - PyTorch version: 2.7.1+cu118
  - CUDA available: True


## 2. Configure Paths and Parameters

In [3]:
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "recipe_generation"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = DATA_DIR / "results" / "pipeline_output"
TEST_IMAGES_DIR = DATA_DIR / "test_images"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Detection settings
INGREDIENTS_CSV = DATA_DIR / "ingredients_vocabulary.csv"
DETECTION_MODE = "multi"  # "single" or "multi"
INGREDIENT_CONFIDENCE_THRESHOLD = 0.15
OBJECT_DETECTION_THRESHOLD = 0.7

# Recipe generation settings
NUM_RECIPES = 5

print("✓ Configuration loaded")
print(f"  - Project root: {PROJECT_ROOT}")
print(f"  - Detection mode: {DETECTION_MODE}")
print(f"  - Number of recipes: {NUM_RECIPES}")

✓ Configuration loaded
  - Project root: c:\Users\Champion\Documents\GitHub\cAIuldron
  - Detection mode: multi
  - Number of recipes: 5


## 3. Load CLIP and DETR Models

In [4]:
if not INGREDIENTS_CSV.exists():
    raise FileNotFoundError(f"Ingredients CSV not found: {INGREDIENTS_CSV}")

df = pd.read_csv(INGREDIENTS_CSV)
INGREDIENT_CANDIDATES = df['Ingredient'].tolist()

print(f"✓ Loaded {len(INGREDIENT_CANDIDATES)} ingredients")

# Load CLIP model
try:
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    print("✓ CLIP model loaded")
except Exception as e:
    print(f"✗ Failed to load CLIP model: {e}")
    clip_model = None
    clip_processor = None

# Load DETR for multi-ingredient detection
if DETECTION_MODE == "multi":
    try:
        detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
        detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
        print("✓ DETR model loaded (multi-ingredient detection)")
    except Exception as e:
        print(f"✗ Failed to load DETR: {e}")
        detr_model = None
        detr_processor = None
else:
    detr_model = None
    detr_processor = None

✓ Loaded 528 ingredients
✓ CLIP model loaded


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✓ DETR model loaded (multi-ingredient detection)


## 4. Multi-Model System for Recipe Generation

In [5]:
from enum import Enum
from dataclasses import dataclass
from typing import Dict, Any

class RecipeModelType(Enum):
    """Available recipe generation models"""
    GPT2 = "gpt2"
    LLAMA_1B = "llama3.2-1b"
    LLAMA_8B_GGUF = "llama3.1-8b-gguf"

@dataclass
class ModelConfig:
    name: str
    display_name: str
    speed: str
    quality: str
    vram_required: str
    description: str

MODEL_INFO = {
    RecipeModelType.GPT2: ModelConfig(
        name="gpt2-finetuned",
        display_name="GPT-2 (Fast)",
        speed="fast",
        quality="good (60-70/100)",
        vram_required="1-2 GB",
        description="Fast but lower quality"
    ),
    RecipeModelType.LLAMA_1B: ModelConfig(
        name="llama3.2-1b-finetuned",
        display_name="Llama 3.2 1B (Recommended)",
        speed="medium",
        quality="excellent (90-95/100)",
        vram_required="4-5 GB",
        description="Recommended: Fast, high quality"
    ),
    RecipeModelType.LLAMA_8B_GGUF: ModelConfig(
        name="llama3.1-8b-gguf",
        display_name="Llama 3.1 8B GGUF (Best)",
        speed="slow",
        quality="excellent (95-100/100)",
        vram_required="6 GB",
        description="Best quality but slower"
    ),
}

RECIPE_MODELS = {}
CURRENT_MODEL_TYPE = RecipeModelType.LLAMA_1B

print("✓ Model configuration loaded")
print(f"  Default: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

✓ Model configuration loaded
  Default: Llama 3.2 1B (Recommended)


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_recipe_model(model_type: RecipeModelType) -> Dict[str, Any]:
    """Load specified recipe generation model"""
    if model_type in RECIPE_MODELS:
        return RECIPE_MODELS[model_type]

    print(f"Loading {MODEL_INFO[model_type].display_name}...")

    if model_type == RecipeModelType.GPT2:
        model_dict = load_gpt2_model()
    elif model_type == RecipeModelType.LLAMA_1B:
        model_dict = load_llama_1b_model()
    elif model_type == RecipeModelType.LLAMA_8B_GGUF:
        model_dict = load_llama_8b_gguf_model()
    
    RECIPE_MODELS[model_type] = model_dict
    return model_dict


def load_gpt2_model() -> Dict[str, Any]:
    """Load fine-tuned GPT-2 model"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    finetuned_dir = MODEL_DIR / "finetuned"
    
    if finetuned_dir.exists():
        tokenizer = GPT2Tokenizer.from_pretrained(str(finetuned_dir))
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2LMHeadModel.from_pretrained(str(finetuned_dir))
        model.to(device)
        model.eval()
        print(f"  ✓ Fine-tuned GPT-2 loaded to {device}")
    else:
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2LMHeadModel.from_pretrained("gpt2")
        model.to(device)
        model.eval()
        print(f"  ⚠️  Base GPT-2 loaded to {device} (fine-tuned not found)")
    
    return {"model": model, "tokenizer": tokenizer, "type": "gpt2", "device": device}


def load_llama_1b_model() -> Dict[str, Any]:
    """Load Llama 3.2 1B fine-tuned model - using 4-bit quantization"""
    base_model_name = "meta-llama/Llama-3.2-1B-Instruct"
    adapter_path = MODEL_DIR / "llama3_1b_finetuned"

    # Use same 4-bit quantization as training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"  ✓ Base model loaded (4-bit), Memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

    if adapter_path.exists():
        model = PeftModel.from_pretrained(model, str(adapter_path))
        model = model.merge_and_unload()
        print("  ✓ LoRA adapter merged - Using your fine-tuned model!")
    else:
        print("  ⚠️  Adapter not found - using base model")

    return {"model": model, "tokenizer": tokenizer, "type": "llama"}


def load_llama_8b_gguf_model() -> Dict[str, Any]:
    """Load Llama 3.1 8B GGUF model - Optimized for RTX 3060 Laptop (4GB VRAM)"""
    try:
        from llama_cpp import Llama
    except ImportError:
        raise ImportError("Install llama-cpp-python: pip install llama-cpp-python")

    model_path = MODEL_DIR / "Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf"
    
    llm = Llama(
        model_path=str(model_path),
        n_gpu_layers=12,        # Optimized for 4GB VRAM (was 20)
        n_ctx=3072,             # Increased context for better recipes (was 2048)
        n_batch=256,            # Reduced to fit 4GB VRAM (was 512)
        n_threads=14,           # Use all CPU cores for offloaded layers
        f16_kv=True,            # Use FP16 for KV cache (saves VRAM)
        verbose=False
    )

    print("  ✓ Llama 3.1 8B GGUF loaded (optimized for RTX 3060 Laptop 4GB)")
    return {"model": llm, "tokenizer": None, "type": "gguf"}

print("✓ Model loading functions defined (with 4-bit quantization for Llama 1B)")

✓ Model loading functions defined (with 4-bit quantization for Llama 1B)


## 5. GPT-2 Recipe Generation Functions

In [7]:
def clean_ingredient_item(item: str) -> Optional[str]:
    """Clean and validate a single ingredient item"""
    item = item.strip()
    if not item:
        return None

    # Remove leading numbers and dots
    item = re.sub(r'^\d+[\.\)]\s*', '', item)

    # Skip if too short
    if len(item) < 3:
        return None

    # Skip nutritional info or metadata
    skip_keywords = ['calories', 'protein', 'fat', 'carbs', 'sodium', 'cholesterol',
                     'serving', 'servings per', 'notes:', 'cooking time', 'prep time']
    if any(kw in item.lower() for kw in skip_keywords):
        return None

    # Skip tags
    if item.startswith('<') and item.endswith('>'):
        return None

    return item


def clean_instruction_step(step: str) -> Optional[str]:
    """Clean and validate instruction step"""
    step = step.strip()
    if not step:
        return None

    # Remove numbers
    step = re.sub(r'^\d+[\.\)]\s*', '', step)
    step = step.strip()
    if not step:
        return None

    # Filter XML/HTML tags
    if '<' in step or '>' in step:
        return None

    # Filter non-Latin characters
    if re.search(r'[　-〿぀-ゟ゠-ヿ一-鿿가-힯฀-๿]', step):
        return None

    # Filter first-person statements
    if any(step.lower().startswith(fp) for fp in ['i ', 'my ', 'we ', "i'm", "i've", "we're"]):
        return None

    # Filter excessive special characters
    if len(re.findall(r'[^a-zA-Z0-9\s\.\,\;\-\(\)]', step)) > 5:
        return None

    # Length limits
    if len(step) < 15 or len(step) > 200:
        return None

    # Nutrition keywords
    skip_kw = ['calories:', 'protein:', 'fat:', 'carbs:', 'sodium:', 'per serving',
               'kcal', 'mg', 'grams', 'fat per', 'calories per']
    if any(kw in step.lower() for kw in skip_kw):
        return None

    # Meta keywords
    meta_kw = ['notes:', 'tips:', 'cooking time:', 'prep time:', 'servings:', 'yields:',
               'recipe can', 'this is', 'very good', 'delicious']
    if any(kw in step.lower() for kw in meta_kw):
        return None

    # Skip if just ingredient listing
    if step.count(';') > 2 or step.count('oz') > 2:
        return None

    # Require cooking verbs
    verbs = ['add', 'mix', 'stir', 'cook', 'bake', 'fry', 'boil', 'simmer', 'saute', 'heat',
             'place', 'cut', 'chop', 'dice', 'slice', 'pour', 'serve', 'season', 'combine',
             'whisk', 'blend', 'reduce', 'drain', 'remove', 'transfer', 'spread', 'cover',
             'preheat', 'prepare', 'arrange', 'garnish', 'sprinkle', 'bring', 'let', 'allow',
             'set', 'top', 'brush', 'toss']
    if not any(v in step.lower() for v in verbs):
        return None

    # Format
    if not step[0].isupper():
        step = step[0].upper() + step[1:]
    if not step.endswith(('.', '!', '?')):
        step += '.'

    return step


def parse_gpt2_recipe_output(text: str, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Parse GPT-2 generated recipe with improved cleaning and validation"""
    recipe = {
        'ingredient': ingredient,
        'recipe_title': f'{cuisine} Style {ingredient.title()}',
        'cuisine': cuisine.lower(),
        'difficulty': difficulty.lower(),
        'cooking_time_minutes': 30,
        'servings': 4,
        'ingredients': [],
        'instructions': [],
        'raw_text': text
    }

    # Extract title
    title_match = re.search(r'<TITLE>\s*(.+?)(?:\n|<)', text)
    if title_match:
        recipe['recipe_title'] = title_match.group(1).strip()

    # Extract cuisine
    cuisine_match = re.search(r'<CUISINE>\s*(.+?)(?:\n|<)', text)
    if cuisine_match:
        recipe['cuisine'] = cuisine_match.group(1).strip().lower()

    # Extract difficulty
    difficulty_match = re.search(r'<DIFFICULTY>\s*(.+?)(?:\n|<)', text)
    if difficulty_match:
        recipe['difficulty'] = difficulty_match.group(1).strip().lower()

    # Extract cooking time
    time_match = re.search(r'<TIME>\s*(\d+)', text)
    if time_match:
        recipe['cooking_time_minutes'] = int(time_match.group(1))
    else:
        time_text_match = re.search(r'(\d+)\s*(?:minute|min)', text.lower())
        if time_text_match:
            recipe['cooking_time_minutes'] = int(time_text_match.group(1))

    # Extract servings
    servings_match = re.search(r'<SERVINGS>\s*(\d+)', text)
    if servings_match:
        recipe['servings'] = int(servings_match.group(1))

    # Extract ingredients with multiple strategies
    ingredients_match = re.search(r'<INGREDIENTS>\s*(.+?)(?:<INSTRUCTIONS>|<|$)', text, re.DOTALL)
    if ingredients_match:
        ingredients_text = ingredients_match.group(1).strip()
        raw_ingredients = []

        # Strategy 1: Split by semicolon
        if ';' in ingredients_text:
            raw_ingredients = re.split(r';', ingredients_text)
        # Strategy 2: Split by newline
        elif '\n' in ingredients_text:
            raw_ingredients = re.split(r'\n', ingredients_text)
        # Strategy 3: Split by numbered list
        else:
            raw_ingredients = re.split(r'\d+[\.\)]\s*', ingredients_text)

        # Clean each ingredient
        for ing in raw_ingredients:
            cleaned = clean_ingredient_item(ing)
            if cleaned:
                recipe['ingredients'].append(cleaned)

    # If structured extraction failed, try to find ingredient patterns
    if not recipe['ingredients']:
        ingredient_pattern = r'(?:^|\n)\s*(?:\d+[\.\)]?\s*)?(\d+(?:/\d+)?\s*(?:c\.|tsp\.|tbsp\.|oz\.|lb\.|cup|teaspoon|tablespoon|ounce|pound)\.?\s+[^;\n]+)'
        ingredient_matches = re.findall(ingredient_pattern, text, re.MULTILINE | re.IGNORECASE)
        for ing in ingredient_matches:
            cleaned = clean_ingredient_item(ing)
            if cleaned:
                recipe['ingredients'].append(cleaned)

    # Ensure at least basic ingredients
    if len(recipe['ingredients']) < 2:
        recipe['ingredients'] = [ingredient, 'salt', 'pepper', 'oil']

    # Limit to reasonable number
    recipe['ingredients'] = recipe['ingredients'][:15]

    # Extract instructions with multiple strategies
    instructions_match = re.search(r'<INSTRUCTIONS>\s*(.+?)(?:<|$)', text, re.DOTALL)
    if instructions_match:
        instructions_text = instructions_match.group(1).strip()
        raw_steps = []

        # Strategy 1: Split by numbered steps
        numbered_split = re.split(r'\n\s*\d+[\.\)]\s*', instructions_text)
        if len(numbered_split) > 1:
            raw_steps = numbered_split
        # Strategy 2: Split by newlines
        elif '\n' in instructions_text:
            raw_steps = re.split(r'\n+', instructions_text)
        # Strategy 3: Split by periods
        else:
            raw_steps = re.split(r'\.\s+', instructions_text)

        # Clean each step
        for step in raw_steps:
            cleaned = clean_instruction_step(step)
            if cleaned:
                recipe['instructions'].append(cleaned)

    # If structured extraction failed, try to find instruction sentences
    if not recipe['instructions']:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        for sentence in sentences:
            cleaned = clean_instruction_step(sentence)
            if cleaned:
                recipe['instructions'].append(cleaned)

    # Ensure at least basic instructions
    if len(recipe['instructions']) < 3:
        recipe['instructions'] = [
            f'Prepare the {ingredient}.',
            'Season with salt and pepper to taste.',
            'Cook according to your preferred method.',
            'Serve hot and enjoy.'
        ]

    # Limit to reasonable number
    recipe['instructions'] = recipe['instructions'][:12]

    return recipe


def generate_recipe_gpt2(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Generate recipe using GPT-2 (your fine-tuned model)"""
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    device = model_dict["device"]
    
    # GPT-2 uses special format markers
    prompt = f"""<INGREDIENT> {ingredient}
<CUISINE> {cuisine}
<TITLE> """
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=350,
            temperature=0.5,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=2
        )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return parse_gpt2_recipe_output(generated_text, ingredient, cuisine, difficulty)

print("✓ GPT-2 recipe generation functions defined (with improved parsing and validation)")

✓ GPT-2 recipe generation functions defined (with improved parsing and validation)


## 6. Llama Recipe Generation Functions

In [8]:
def generate_recipe_from_ingredients(model, tokenizer, ingredients_str, cuisine=None):
    """
    Generate recipe from ingredients using fine-tuned Llama model
    This is the generation function used during training
    """
    cuisine_hint = f" ({cuisine} style)" if cuisine else ""
    
    # Llama 3.2 uses special conversation format
    # STRENGTHENED PROMPT: Include concrete example with actual numbers
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. You MUST follow this EXACT format for EVERY recipe.

REQUIRED FORMAT (copy this structure EXACTLY):

# [Recipe Title]

**Prep Time**: 15 minutes
**Cook Time**: 30 minutes  
**Total Time**: 45 minutes
**Servings**: 4

## Ingredients
- ingredient 1
- ingredient 2

## Instructions
1. Step 1
2. Step 2

CRITICAL RULES:
1. ALWAYS include Prep Time, Cook Time, Total Time, Servings (no exceptions!)
2. Time format MUST be: "**Prep Time**: [number] minutes"
3. Use realistic cooking times based on recipe complexity
4. Start immediately with "# " followed by recipe title<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a delicious recipe using these ingredients: {ingredients_str}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1
    )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    response = full_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    response = response.split("<|eot_id|>")[0].strip()
    
    return response


def estimate_times_with_llm(model, tokenizer, recipe_text: str, difficulty: str) -> Dict[str, int]:
    """
    Use Llama to estimate Prep/Cook/Total time for a recipe
    """
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a cooking time expert.<|eot_id|><|start_header_id|>user<|end_header_id|>

Estimate realistic cooking times for this {difficulty} recipe. Reply ONLY with numbers in this format:
Prep: [X]
Cook: [Y]
Total: [Z]

Recipe:
{recipe_text[:300]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Prep: """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("assistant")[-1].strip()
    
    # Extract numbers
    prep_match = re.search(r'Prep:\s*(\d+)', response, re.IGNORECASE)
    cook_match = re.search(r'Cook:\s*(\d+)', response, re.IGNORECASE)
    total_match = re.search(r'Total:\s*(\d+)', response, re.IGNORECASE)
    
    prep_time = int(prep_match.group(1)) if prep_match else 15
    cook_time = int(cook_match.group(1)) if cook_match else 25
    total_time = int(total_match.group(1)) if total_match else (prep_time + cook_time)
    
    return {"prep": prep_time, "cook": cook_time, "total": total_time}


def estimate_times_with_gguf(llm, recipe_text: str, difficulty: str) -> Dict[str, int]:
    """
    Use GGUF Llama to estimate Prep/Cook/Total time for a recipe
    """
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a cooking time expert.<|eot_id|><|start_header_id|>user<|end_header_id|>

Estimate realistic cooking times for this {difficulty} recipe. Reply ONLY with numbers in this format:
Prep: [X]
Cook: [Y]
Total: [Z]

Recipe:
{recipe_text[:300]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Prep: """
    
    output = llm(
        prompt,
        max_tokens=50,
        temperature=0.3,
        top_p=0.9,
        stop=["<|eot_id|>", "\n\n"]
    )
    
    response = output['choices'][0]['text']
    
    # Extract numbers
    prep_match = re.search(r'Prep:\s*(\d+)', response, re.IGNORECASE)
    cook_match = re.search(r'Cook:\s*(\d+)', response, re.IGNORECASE)
    total_match = re.search(r'Total:\s*(\d+)', response, re.IGNORECASE)
    
    prep_time = int(prep_match.group(1)) if prep_match else 15
    cook_time = int(cook_match.group(1)) if cook_match else 25
    total_time = int(total_match.group(1)) if total_match else (prep_time + cook_time)
    
    return {"prep": prep_time, "cook": cook_time, "total": total_time}


def validate_time_format(response: str) -> Dict[str, bool]:
    """
    Validate that time fields exist and are in correct format
    Returns dict with validation results
    """
    validation = {
        'has_prep_time': False,
        'has_cook_time': False,
        'has_total_time': False,
        'has_servings': False,
        'prep_format_correct': False,
        'cook_format_correct': False,
        'total_format_correct': False,
        'servings_format_correct': False
    }
    
    # Check for existence and correct format: "**Prep Time**: [number] minutes"
    prep_pattern = r'\*\*Prep Time\*\*:\s*(\d+)\s*minutes?'
    cook_pattern = r'\*\*Cook Time\*\*:\s*(\d+)\s*minutes?'
    total_pattern = r'\*\*Total Time\*\*:\s*(\d+)\s*minutes?'
    servings_pattern = r'\*\*Servings\*\*:\s*(\d+)'
    
    prep_match = re.search(prep_pattern, response, re.IGNORECASE)
    cook_match = re.search(cook_pattern, response, re.IGNORECASE)
    total_match = re.search(total_pattern, response, re.IGNORECASE)
    servings_match = re.search(servings_pattern, response, re.IGNORECASE)
    
    if prep_match:
        validation['has_prep_time'] = True
        validation['prep_format_correct'] = True
    
    if cook_match:
        validation['has_cook_time'] = True
        validation['cook_format_correct'] = True
    
    if total_match:
        validation['has_total_time'] = True
        validation['total_format_correct'] = True
    
    if servings_match:
        validation['has_servings'] = True
        validation['servings_format_correct'] = True
    
    return validation


def ensure_time_fields_with_llm(response: str, difficulty: str, model=None, tokenizer=None, llm=None) -> str:
    """
    Ensure recipe has Prep Time, Cook Time, Total Time, Servings with correct format
    Only uses LLM estimation if fields are completely missing
    """
    # First, validate the current format
    validation = validate_time_format(response)
    
    # If all fields exist in correct format, return as-is (best case - no fallback needed!)
    all_valid = (validation['prep_format_correct'] and 
                 validation['cook_format_correct'] and 
                 validation['total_format_correct'] and 
                 validation['servings_format_correct'])
    
    if all_valid:
        return response
    
    # If fields are missing, use LLM to estimate times
    if model and tokenizer:
        times = estimate_times_with_llm(model, tokenizer, response, difficulty)
    elif llm:
        times = estimate_times_with_gguf(llm, response, difficulty)
    else:
        # Ultimate fallback (shouldn't happen)
        times = {"prep": 15, "cook": 25, "total": 40}
    
    # Insert missing fields after title (using CORRECT format)
    lines = response.split('\n')
    new_lines = []
    inserted = False
    
    for line in lines:
        new_lines.append(line)
        
        # Insert after first # heading
        if not inserted and line.startswith('#') and not line.startswith('##'):
            new_lines.append('')
            if not validation['prep_format_correct']:
                new_lines.append(f'**Prep Time**: {times["prep"]} minutes')
            if not validation['cook_format_correct']:
                new_lines.append(f'**Cook Time**: {times["cook"]} minutes')
            if not validation['total_format_correct']:
                new_lines.append(f'**Total Time**: {times["total"]} minutes')
            if not validation['servings_format_correct']:
                new_lines.append(f'**Servings**: 4')
            new_lines.append('')
            inserted = True
    
    return '\n'.join(new_lines)


def parse_llama_recipe_output(response: str, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None, model=None, tokenizer=None) -> Dict[str, Any]:
    """
    Format Llama output - extract title, time, servings from content
    Uses Llama to estimate missing times only if needed
    """
    # Ensure time fields exist with correct format (minimal fallback)
    response = ensure_time_fields_with_llm(response, difficulty, model, tokenizer, llm=None)
    
    lines = response.strip().split('\n')
    
    # Extract title (first # heading)
    title = "Recipe"
    for line in lines:
        if line.startswith('#') and not line.startswith('##'):
            title = line.strip('# ').strip()
            break
    
    # Extract cooking time from Markdown content
    cooking_time = 30  # default
    time_patterns = [
        r'(?:Cooking Time|Cook Time|Total Time|Time)[\s:*]+(\d+)[\s-]*(?:min|minute)',
        r'\*\*(?:Cooking Time|Cook Time|Total Time|Time)\*\*[\s:]+(\d+)',
        r'(\d+)[\s-]*(?:min|minute)(?:ute)?s?\s+(?:cooking|cook|total)',
    ]
    
    response_lower = response.lower()
    for pattern in time_patterns:
        match = re.search(pattern, response_lower, re.IGNORECASE)
        if match:
            cooking_time = int(match.group(1))
            break
    
    # Extract servings: priority: nutrition data > Markdown content > default
    servings = 4  # default
    
    # Priority 1: Use nutrition data if available
    if nutrition_data and 'servings' in nutrition_data:
        servings = nutrition_data['servings']
    else:
        # Priority 2: Extract from Markdown content
        servings_patterns = [
            r'(?:Servings|Serves|Yield)[\s:*]+(\d+)',
            r'\*\*(?:Servings|Serves|Yield)\*\*[\s:]+(\d+)',
            r'(?:Makes|Yields)\s+(\d+)\s+(?:serving|portion)',
        ]
        
        for pattern in servings_patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                servings = int(match.group(1))
                break
    
    # Keep complete Markdown content
    return {
        "ingredient": ingredient,
        "recipe_title": title,
        "cuisine": cuisine,
        "difficulty": difficulty,
        "cooking_time_minutes": cooking_time,
        "servings": servings,
        "raw_markdown": response,
        "ingredients": [ingredient],
        "instructions": [response]
    }


def parse_gguf_recipe_output(response: str, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None, llm=None) -> Dict[str, Any]:
    """
    Format GGUF output - extract title, time, servings from content
    Uses Llama to estimate missing times only if needed
    """
    # Ensure time fields exist with correct format (minimal fallback)
    response = ensure_time_fields_with_llm(response, difficulty, model=None, tokenizer=None, llm=llm)
    
    lines = response.strip().split('\n')
    
    # Extract title (first # heading)
    title = "Recipe"
    for line in lines:
        if line.startswith('#') and not line.startswith('##'):
            title = line.strip('# ').strip()
            break
    
    # Extract cooking time from Markdown content
    cooking_time = 30  # default
    time_patterns = [
        r'(?:Cooking Time|Cook Time|Total Time|Time)[\s:*]+(\d+)[\s-]*(?:min|minute)',
        r'\*\*(?:Cooking Time|Cook Time|Total Time|Time)\*\*[\s:]+(\d+)',
        r'(\d+)[\s-]*(?:min|minute)(?:ute)?s?\s+(?:cooking|cook|total)',
    ]
    
    response_lower = response.lower()
    for pattern in time_patterns:
        match = re.search(pattern, response_lower, re.IGNORECASE)
        if match:
            cooking_time = int(match.group(1))
            break
    
    # Extract servings: priority: nutrition data > Markdown content > default
    servings = 4  # default
    
    if nutrition_data and 'servings' in nutrition_data:
        servings = nutrition_data['servings']
    else:
        servings_patterns = [
            r'(?:Servings|Serves|Yield)[\s:*]+(\d+)',
            r'\*\*(?:Servings|Serves|Yield)\*\*[\s:]+(\d+)',
            r'(?:Makes|Yields)\s+(\d+)\s+(?:serving|portion)',
        ]
        
        for pattern in servings_patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                servings = int(match.group(1))
                break
    
    return {
        "ingredient": ingredient,
        "recipe_title": title,
        "cuisine": cuisine,
        "difficulty": difficulty,
        "cooking_time_minutes": cooking_time,
        "servings": servings,
        "raw_markdown": response,
        "ingredients": [ingredient],
        "instructions": [response]
    }


def generate_recipe_llama(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None) -> Dict[str, Any]:
    """Generate recipe using Llama model (your fine-tuned model)"""
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    
    # Generate recipe
    response = generate_recipe_from_ingredients(model, tokenizer, ingredient, cuisine)
    
    # Parse and ensure times (pass model for time estimation if needed)
    return parse_llama_recipe_output(response, ingredient, cuisine, difficulty, nutrition_data, model, tokenizer)


def generate_recipe_gguf(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None) -> Dict[str, Any]:
    """Generate recipe using GGUF model (Llama 3.1 8B)"""
    llm = model_dict["model"]
    cuisine_hint = f" ({cuisine} style)" if cuisine and cuisine != "any" else ""
    
    # STRENGTHENED PROMPT: Include concrete example with actual numbers
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. You MUST follow this EXACT format for EVERY recipe.

REQUIRED FORMAT (copy this structure EXACTLY):

# [Recipe Title]

**Prep Time**: 15 minutes
**Cook Time**: 30 minutes
**Total Time**: 45 minutes
**Servings**: 4

## Ingredients
- ingredient 1
- ingredient 2

## Instructions
1. Step 1
2. Step 2

CRITICAL RULES:
1. ALWAYS include Prep Time, Cook Time, Total Time, Servings (no exceptions!)
2. Time format MUST be: "**Prep Time**: [number] minutes"
3. Use realistic cooking times based on recipe complexity
4. Start immediately with "# " followed by recipe title<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a {difficulty} recipe using: {ingredient}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.7,
        top_p=0.9,
        stop=["<|eot_id|>"]
    )
    
    response = output['choices'][0]['text']
    
    # Parse and ensure times (pass llm for time estimation if needed)
    return parse_gguf_recipe_output(response, ingredient, cuisine, difficulty, nutrition_data, llm)

print("✓ Llama recipe generation functions defined (strengthened prompts + format validation)")

✓ Llama recipe generation functions defined (strengthened prompts + format validation)


## 7. Model Router

In [9]:
def generate_recipe_with_selected_model(
    ingredient: str,
    cuisine: str = "any",
    difficulty: str = "medium",
    model_type: RecipeModelType = None,
    nutrition_data: dict = None
) -> Dict[str, Any]:
    """
    Generate recipe using selected model
    
    This is the main entry point for recipe generation.
    It routes to the appropriate generation function based on model type.
    """
    if model_type is None:
        model_type = CURRENT_MODEL_TYPE
    
    # Load model (or use cached)
    model_dict = load_recipe_model(model_type)
    
    # Route to appropriate generation function
    if model_dict["type"] == "gpt2":
        return generate_recipe_gpt2(model_dict, ingredient, cuisine, difficulty)
    elif model_dict["type"] == "llama":
        return generate_recipe_llama(model_dict, ingredient, cuisine, difficulty, nutrition_data)
    elif model_dict["type"] == "gguf":
        return generate_recipe_gguf(model_dict, ingredient, cuisine, difficulty, nutrition_data)
    else:
        raise ValueError(f"Unknown model type: {model_dict.get('type')}")

print("✓ Model router defined (with nutrition data support)")

✓ Model router defined (with nutrition data support)


In [10]:
# Interactive Model Selector
import ipywidgets as widgets
from IPython.display import display

model_selector = widgets.Dropdown(
    options=[
        ('GPT-2 (Fast)', RecipeModelType.GPT2),
        ('Llama 3.2 1B (Recommended)', RecipeModelType.LLAMA_1B),
        ('Llama 3.1 8B GGUF (Best Quality)', RecipeModelType.LLAMA_8B_GGUF)
    ],
    value=RecipeModelType.LLAMA_1B,
    description='Model:'
)

info_button = widgets.Button(description='Show Info', button_style='info')
info_output = widgets.Output()

def show_model_info(b):
    with info_output:
        info_output.clear_output()
        print("Available Models:\n")
        for model_type, config in MODEL_INFO.items():
            print(f"{config.display_name}")
            print(f"  Speed: {config.speed} | Quality: {config.quality}")
            print(f"  VRAM: {config.vram_required}\n")

info_button.on_click(show_model_info)

def update_current_model(change):
    global CURRENT_MODEL_TYPE
    CURRENT_MODEL_TYPE = change['new']
    print(f"Switched to: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

model_selector.observe(update_current_model, names='value')

display(widgets.VBox([
    widgets.Label('Select Recipe Generation Model:'),
    model_selector,
    info_button,
    info_output
]))

print(f"Current model: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

Current model: Llama 3.2 1B (Recommended)


## 8. Load Nutrition Database

In [11]:
NUTRITION_JSON = DATA_DIR / "nutrition_lookup_full.json"

if NUTRITION_JSON.exists():
    with open(NUTRITION_JSON, 'r', encoding='utf-8') as f:
        NUTRITION_DB = json.load(f)
    print(f"✓ Nutrition database loaded: {len(NUTRITION_DB)} ingredients")
else:
    print(f"⚠ Nutrition database not found, using fallback...")
    NUTRITION_DB = {
        'chicken breast': {'calories': 165, 'protein_g': 31, 'fat_g': 3.6, 'carbs_g': 0},
        'chicken': {'calories': 239, 'protein_g': 27, 'fat_g': 14, 'carbs_g': 0},
        'beef': {'calories': 250, 'protein_g': 26, 'fat_g': 15, 'carbs_g': 0},
        'salmon': {'calories': 208, 'protein_g': 20, 'fat_g': 13, 'carbs_g': 0},
        'tomato': {'calories': 18, 'protein_g': 0.9, 'fat_g': 0.2, 'carbs_g': 3.9},
        'potato': {'calories': 77, 'protein_g': 2, 'fat_g': 0.1, 'carbs_g': 17},
        'egg': {'calories': 155, 'protein_g': 13, 'fat_g': 11, 'carbs_g': 1.1},
        'rice': {'calories': 130, 'protein_g': 2.7, 'fat_g': 0.3, 'carbs_g': 28},
    }
    print(f"✓ Fallback database loaded: {len(NUTRITION_DB)} ingredients")

TYPICAL_WEIGHTS = {
    'chicken breast': 200, 'chicken': 150, 'beef': 200,
    'salmon': 150, 'tomato': 120, 'potato': 180,
    'egg': 50, 'rice': 150
}

✓ Nutrition database loaded: 525 ingredients


## 9. Detection and Helper Functions

In [12]:
def detect_ingredient_clip(image_path: str, confidence_threshold: float = 0.15) -> Dict:
    """Detect ingredient using CLIP (single-ingredient mode)"""
    image = Image.open(image_path).convert('RGB')
    
    inputs = clip_processor(
        text=INGREDIENT_CANDIDATES,
        images=image,
        return_tensors="pt",
        padding=True
    )
    
    with torch.no_grad():
        outputs = clip_model(**inputs)
    
    probs = outputs.logits_per_image.softmax(dim=1)[0]
    top_prob, top_idx = probs.max(0)
    
    if top_prob.item() < confidence_threshold:
        return None
    
    img_width, img_height = image.size
    
    return {
        'class': INGREDIENT_CANDIDATES[top_idx],
        'confidence': top_prob.item(),
        'width': img_width * 0.6,
        'height': img_height * 0.6,
        'x': img_width / 2,
        'y': img_height / 2,
        'detection_method': 'CLIP_single'
    }


def detect_multiple_ingredients_clip(image_path: str, 
                                     object_threshold: float = 0.7,
                                     ingredient_threshold: float = 0.15) -> List[Dict]:
    """Detect multiple ingredients using DETR + CLIP"""
    image = Image.open(image_path).convert('RGB')
    
    inputs = detr_processor(images=image, return_tensors="pt")
    outputs = detr_model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]])
    results = detr_processor.post_process_object_detection(
        outputs, 
        target_sizes=target_sizes, 
        threshold=object_threshold
    )[0]
    
    detected_ingredients = []
    
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box_coords = [int(i) for i in box.tolist()]
        x1, y1, x2, y2 = box_coords
        
        cropped = image.crop((x1, y1, x2, y2))
        
        clip_inputs = clip_processor(
            text=INGREDIENT_CANDIDATES,
            images=cropped,
            return_tensors="pt",
            padding=True
        )
        
        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
        
        probs = clip_outputs.logits_per_image.softmax(dim=1)[0]
        top_prob, top_idx = probs.max(0)
        
        if top_prob.item() >= ingredient_threshold:
            detected_ingredients.append({
                'class': INGREDIENT_CANDIDATES[top_idx],
                'confidence': top_prob.item(),
                'width': x2 - x1,
                'height': y2 - y1,
                'x': (x1 + x2) / 2,
                'y': (y1 + y2) / 2,
                'bbox': box_coords,
                'detection_method': 'DETR+CLIP_multi'
            })
    
    return detected_ingredients


def generate_diverse_prompts(ingredient: str, num_recipes: int = 5) -> List[Dict]:
    """Generate diverse cuisine prompts"""
    configurations = [
        {'cuisine': 'Asian', 'difficulty': 'beginner'},
        {'cuisine': 'Western', 'difficulty': 'beginner'},
        {'cuisine': 'Fusion', 'difficulty': 'intermediate'},
        {'cuisine': 'Mediterranean', 'difficulty': 'beginner'},
        {'cuisine': 'any', 'difficulty': 'intermediate'},
    ]
    
    prompts = []
    for i in range(min(num_recipes, len(configurations))):
        config = configurations[i]
        prompts.append({
            'ingredient': ingredient,
            'cuisine': config['cuisine'],
            'difficulty': config['difficulty'],
            'recipe_index': i + 1
        })
    
    return prompts


def estimate_nutrition(ingredient: str, bbox_width: int, bbox_height: int,
                      image_width: int = 640, image_height: int = 640) -> Dict:
    """Estimate nutrition from bounding box"""
    ingredient_lower = ingredient.lower()
    
    typical_weight = TYPICAL_WEIGHTS.get(ingredient_lower, 150)
    bbox_area = bbox_width * bbox_height
    image_area = image_width * image_height
    area_ratio = bbox_area / image_area
    size_multiplier = (area_ratio / 0.25) ** 0.7
    estimated_weight = typical_weight * size_multiplier
    
    if any(m in ingredient_lower for m in ['chicken', 'beef', 'pork', 'salmon', 'fish']):
        serving_size = 120
    elif any(v in ingredient_lower for v in ['potato', 'tomato', 'vegetable']):
        serving_size = 100
    else:
        serving_size = 100
    
    servings = max(1, round(estimated_weight / serving_size * 2) / 2)
    g_per_serving = estimated_weight / servings
    
    nutrition_base = None
    if ingredient in NUTRITION_DB:
        nutrition_base = NUTRITION_DB[ingredient]
    else:
        for key in NUTRITION_DB.keys():
            if key.lower() in ingredient_lower or ingredient_lower in key.lower():
                nutrition_base = NUTRITION_DB[key]
                break
    
    if not nutrition_base:
        return {'success': False, 'error': f'No nutrition data for {ingredient}'}
    
    multiplier = g_per_serving / 100
    calories = nutrition_base['calories'] * multiplier
    
    return {
        'success': True,
        'weight_g': round(estimated_weight, 1),
        'servings': int(servings) if servings.is_integer() else servings,
        'per_serving': {
            'weight_g': round(g_per_serving, 1),
            'calories': round(calories, 0),
            'calories_range': f"{round(calories*0.8, 0):.0f}-{round(calories*1.2, 0):.0f} kcal",
            'protein_g': round(nutrition_base['protein_g'] * multiplier, 1),
            'fat_g': round(nutrition_base['fat_g'] * multiplier, 1),
            'carbs_g': round(nutrition_base['carbs_g'] * multiplier, 1)
        }
    }

print("✓ Helper functions defined")

✓ Helper functions defined


## 10. Main Pipeline Function

In [13]:
def process_ingredient_photo(image_path: str, 
                            confidence_threshold: Optional[float] = None,
                            detection_mode: Optional[str] = None,
                            verbose: bool = True) -> Dict:
    """Complete pipeline: Photo → Recipes with Nutrition"""
    start_time = time.time()
    
    conf_threshold = confidence_threshold if confidence_threshold is not None else INGREDIENT_CONFIDENCE_THRESHOLD
    mode = detection_mode if detection_mode is not None else DETECTION_MODE
    
    if verbose:
        print("\n" + "="*80)
        print(f"RECIPE GENERATION PIPELINE ({MODEL_INFO[CURRENT_MODEL_TYPE].display_name})")
        print("="*80)
    
    # Step 1: Detect ingredients
    try:
        if mode == "single":
            primary = detect_ingredient_clip(image_path, conf_threshold)
            if not primary:
                return {'success': False, 'error': 'No ingredient detected'}
            detected = [primary]
        else:
            detected = detect_multiple_ingredients_clip(
                image_path,
                object_threshold=OBJECT_DETECTION_THRESHOLD,
                ingredient_threshold=conf_threshold
            )
            if not detected:
                return {'success': False, 'error': 'No ingredients detected'}
        
        if verbose:
            print(f"✓ Found {len(detected)} ingredient(s)")
    except Exception as e:
        return {'success': False, 'error': f'Detection failed: {e}'}
    
    # Step 2: Consolidate ingredients
    ingredient_map = {}
    for detection in detected:
        ing_name = detection['class']
        if ing_name not in ingredient_map:
            ingredient_map[ing_name] = {
                'name': ing_name,
                'confidence': detection['confidence'],
                'count': 1,
                'total_area': detection['width'] * detection['height']
            }
        else:
            ingredient_map[ing_name]['confidence'] = max(
                ingredient_map[ing_name]['confidence'],
                detection['confidence']
            )
            ingredient_map[ing_name]['count'] += 1
            ingredient_map[ing_name]['total_area'] += detection['width'] * detection['height']
    
    unique_ingredients = list(ingredient_map.keys())
    combined_ingredient = " and ".join(unique_ingredients)
    
    # Step 3: Calculate nutrition (before generating recipes)
    primary_ing = max(ingredient_map.items(), key=lambda x: x[1]['total_area'])
    primary_name = primary_ing[0]
    primary_area = primary_ing[1]['total_area']
    
    estimated_width = int(primary_area ** 0.5)
    estimated_height = int(primary_area ** 0.5)
    
    nutrition = estimate_nutrition(primary_name, estimated_width, estimated_height)
    nutrition_data = nutrition if nutrition['success'] else None
    
    # Step 4: Generate recipes (with nutrition data)
    if verbose:
        print(f"Generating {NUM_RECIPES} recipes...")
    
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    for prompt in prompts:
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty'],
            nutrition_data=nutrition_data
        )
        recipes.append(recipe)
    
    # Step 5: Add nutrition info to recipes
    for recipe in recipes:
        recipe['nutrition'] = nutrition.get('per_serving', {}) if nutrition['success'] else None
    
    elapsed_time = time.time() - start_time
    
    if verbose:
        print(f"✓ Complete in {elapsed_time:.2f}s")
    
    return {
        'success': True,
        'ingredient': {
            'name': combined_ingredient,
            'unique_ingredients': unique_ingredients,
            'primary_ingredient': primary_name,
            'confidence': ingredient_map[primary_name]['confidence']
        },
        'nutrition': nutrition if nutrition['success'] else None,
        'recipes': recipes,
        'num_recipes': len(recipes),
        'processing_time_seconds': elapsed_time
    }

print("✓ Pipeline function defined (with smart servings extraction)")

✓ Pipeline function defined (with smart servings extraction)


## 11. Recipe JSON Auto-Fix Functions

In [14]:
def extract_title_from_markdown(markdown_text: str) -> str:
    """Extract real title from Markdown content"""
    if not markdown_text:
        return "Untitled Recipe"
    
    lines = markdown_text.split('\n')
    for line in lines:
        # Find first # heading as title
        if line.strip().startswith('#') and not line.strip().startswith('##'):
            title = line.strip('#').strip()
            # Skip if title looks like an ingredient (contains measurements)
            if re.search(r'^\d+(\s+\d+/\d+|/\d+|\.\d+)?\s+(c\.|Tbsp\.|tsp\.|lb\.|oz\.)', title):
                continue
            return title
    
    return "Untitled Recipe"


def fix_recipe_json(recipe: Dict) -> Dict:
    """
    Fix common issues in generated recipes:
    - Fix wrong titles (ingredient names → real titles)
    - Fix ingredient format issues
    - Detect truncated content
    """
    fixed = recipe.copy()
    
    # 1. Fix wrong titles (e.g., "1 1/2 c. all-purpose flour")
    current_title = fixed.get('recipe_title', '')
    # Match "1 c.", "1/2 c.", "1 1/2 c.", "1.5 Tbsp." etc.
    is_ingredient_format = bool(re.search(r'^\d+(\s+\d+/\d+|/\d+|\.\d+)?\s+(c\.|Tbsp\.|tsp\.|lb\.|oz\.)', current_title))
    
    if is_ingredient_format:
        # Try to extract real title from markdown
        new_title = None
        if fixed.get('raw_markdown'):
            new_title = extract_title_from_markdown(fixed['raw_markdown'])
        
        # Generate title if extraction failed
        if not new_title or new_title == "Untitled Recipe":
            ingredient = fixed.get('ingredient', 'Ingredient')
            cuisine = fixed.get('cuisine', 'any').capitalize()
            if cuisine == "Any":
                cuisine = "Simple"
            fixed['recipe_title'] = f"{cuisine} {ingredient} Recipe"
        else:
            fixed['recipe_title'] = new_title
    
    # 2. Fix ingredient list format (# 1 cup → - 1 cup)
    if fixed.get('raw_markdown'):
        raw_md = fixed['raw_markdown']
        if re.search(r'^#\s+\d+', raw_md, re.MULTILINE):
            fixed_md = re.sub(r'^#\s+', '- ', raw_md, flags=re.MULTILINE)
            fixed['raw_markdown'] = fixed_md
    
    # 3. Detect truncated content
    if fixed.get('raw_markdown'):
        raw_md = fixed['raw_markdown'].strip()
        if raw_md.endswith(('Use', 'Add', 'Pour', 'Mix', 'Stir')):
            fixed['_warning'] = 'Content may be truncated'
    
    return fixed

print("✓ Recipe auto-fix functions defined")

✓ Recipe auto-fix functions defined


## 12. Gradio Interface

In [15]:
try:
    import gradio as gr
    print("✓ Gradio already installed")
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
    import gradio as gr
    print("✓ Gradio installed")

✓ Gradio already installed


In [16]:
from datetime import datetime

def save_recipe_to_json(recipe: Dict, recipe_id: int, session_id: str = None) -> str:
    """Save recipe to JSON file"""
    if session_id is None:
        session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    output_dir = RESULTS_DIR / "recipes_json" / session_id
    output_dir.mkdir(parents=True, exist_ok=True)
    
    filename = f"recipe_{recipe_id:02d}.json"
    filepath = output_dir / filename
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(recipe, f, ensure_ascii=False, indent=2)
    
    return str(filepath)


def gradio_detect_ingredients(image, confidence_threshold):
    """Stage 1: Detect ingredients and calculate nutrition"""
    if image is None:
        return "⚠️ Please upload an image first.", "", "{}"
    
    temp_path = RESULTS_DIR / "temp_upload.jpg"
    image.save(temp_path)
    
    start_time = time.time()
    conf_threshold = confidence_threshold if confidence_threshold is not None else INGREDIENT_CONFIDENCE_THRESHOLD
    mode = DETECTION_MODE
    
    try:
        if mode == "single":
            primary = detect_ingredient_clip(str(temp_path), conf_threshold)
            if not primary:
                return "❌ No ingredient detected", "", "{}"
            detected = [primary]
        else:
            detected = detect_multiple_ingredients_clip(
                str(temp_path),
                object_threshold=OBJECT_DETECTION_THRESHOLD,
                ingredient_threshold=conf_threshold
            )
            if not detected:
                return "❌ No ingredients detected", "", "{}"
    except Exception as e:
        return f"❌ Detection failed: {e}", "", "{}"
    
    # Consolidate
    ingredient_map = {}
    for detection in detected:
        ing_name = detection['class']
        if ing_name not in ingredient_map:
            ingredient_map[ing_name] = {
                'name': ing_name,
                'confidence': detection['confidence'],
                'count': 1,
                'total_area': detection['width'] * detection['height']
            }
        else:
            ingredient_map[ing_name]['confidence'] = max(
                ingredient_map[ing_name]['confidence'],
                detection['confidence']
            )
            ingredient_map[ing_name]['count'] += 1
            ingredient_map[ing_name]['total_area'] += detection['width'] * detection['height']
    
    unique_ingredients = list(ingredient_map.keys())
    combined_ingredient = " and ".join(unique_ingredients)
    
    # Calculate nutrition
    primary_ing = max(ingredient_map.items(), key=lambda x: x[1]['total_area'])
    primary_name = primary_ing[0]
    primary_area = primary_ing[1]['total_area']
    
    estimated_width = int(primary_area ** 0.5)
    estimated_height = int(primary_area ** 0.5)
    nutrition = estimate_nutrition(primary_name, estimated_width, estimated_height)
    
    elapsed_time = time.time() - start_time
    
    detection_result = f"""# 🔍 Detection Results

**Ingredients**: {combined_ingredient}
**Primary**: {primary_name}
**Confidence**: {ingredient_map[primary_name]['confidence']:.1%}
**Time**: {elapsed_time:.2f}s

✅ Detection complete! Click 'Generate Recipes' to continue.
"""
    
    if nutrition['success']:
        nutrition_result = f"""# 🥗 Nutrition Information

**Ingredient**: {primary_name}
**Weight**: {nutrition['weight_g']}g
**Servings**: {nutrition['servings']}

### Per Serving ({nutrition['per_serving']['weight_g']}g)
- Calories: {nutrition['per_serving']['calories']:.0f} kcal
- Protein: {nutrition['per_serving']['protein_g']}g
- Fat: {nutrition['per_serving']['fat_g']}g
- Carbs: {nutrition['per_serving']['carbs_g']}g
"""
    else:
        nutrition_result = "⚠️ Nutrition data not available"
    
    detection_data = {
        'combined_ingredient': combined_ingredient,
        'unique_ingredients': unique_ingredients,
        'primary_name': primary_name,
        'nutrition': nutrition if nutrition['success'] else None
    }
    
    return detection_result, nutrition_result, json.dumps(detection_data)


def gradio_generate_recipes(detection_data_json, selected_model):
    """
    Stage 2: Generate recipes with detailed time information in content
    Flow: Generate (with nutrition data) → Save → Read → Fix → Re-save → Display
    """
    if not detection_data_json or detection_data_json == "{}":
        yield "⚠️ Please detect ingredients first"
        return
    
    try:
        detection_data = json.loads(detection_data_json)
    except:
        yield "❌ Invalid detection data"
        return
    
    combined_ingredient = detection_data['combined_ingredient']
    nutrition_data = detection_data.get('nutrition')
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Parse selected model
    model_type = RecipeModelType.LLAMA_1B
    if "GPT-2" in selected_model:
        model_type = RecipeModelType.GPT2
    elif "Llama 3.2 1B" in selected_model:
        model_type = RecipeModelType.LLAMA_1B
    elif "Llama 3.1 8B" in selected_model:
        model_type = RecipeModelType.LLAMA_8B_GGUF
    
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

⏳ Step 1/3: Generating recipes with Prep Time, Cook Time, Total Time...
"""
    
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    # Step 1: Generate & Save (with nutrition data)
    for idx, prompt in enumerate(prompts, 1):
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty'],
            model_type=model_type,
            nutrition_data=nutrition_data
        )
        
        if detection_data.get('nutrition'):
            recipe['nutrition'] = detection_data['nutrition'].get('per_serving', {})
        
        recipe['session_id'] = session_id
        recipe['model_used'] = selected_model
        recipe['model_type'] = model_type.value
        recipe['generated_at'] = datetime.now().isoformat()
        
        json_path = save_recipe_to_json(recipe, idx, session_id)
        recipe['json_path'] = json_path
        recipes.append(recipe)
        
        yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

⏳ Step 1/3: Generated {idx}/{NUM_RECIPES} recipes...
"""
    
    # Step 2: Read & Fix & Re-save
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

✅ Step 1/3: Complete
⏳ Step 2/3: Auto-fixing recipes...
"""
    
    session_dir = RESULTS_DIR / "recipes_json" / session_id
    fixed_recipes = []
    
    for idx in range(1, len(recipes) + 1):
        json_file = session_dir / f"recipe_{idx:02d}.json"
        
        with open(json_file, 'r', encoding='utf-8') as f:
            recipe = json.load(f)
        
        fixed_recipe = fix_recipe_json(recipe)
        
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(fixed_recipe, f, ensure_ascii=False, indent=2)
        
        fixed_recipes.append(fixed_recipe)
    
    # Step 3: Display
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

✅ Step 1/3: Generated {len(recipes)} recipes
✅ Step 2/3: Auto-fixed all recipes
⏳ Step 3/3: Displaying results...
"""
    
    final_result = f"""# 🍳 Generated & Fixed Recipes ({len(fixed_recipes)})

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

---

"""
    
    for idx, recipe in enumerate(fixed_recipes, 1):
        # Only show title, cuisine, difficulty
        final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')}

"""
        
        # Display content (includes Prep Time, Cook Time, Total Time, Servings)
        if recipe.get('raw_markdown'):
            raw_md = recipe['raw_markdown']
            lines = raw_md.split('\n')
            content_lines = []
            skip_first_heading = False
            
            for line in lines:
                if not skip_first_heading and line.startswith('#') and not line.startswith('##'):
                    skip_first_heading = True
                    continue
                content_lines.append(line)
            
            cleaned_markdown = '\n'.join(content_lines)
            final_result += cleaned_markdown + "\n\n"
        else:
            # GPT-2 format
            if recipe.get('ingredients'):
                final_result += "### Ingredients\n"
                for ing in recipe['ingredients'][:8]:
                    final_result += f"- {ing}\n"
                final_result += "\n"
            
            if recipe.get('instructions'):
                final_result += "### Instructions\n"
                for step_idx, step in enumerate(recipe['instructions'][:6], 1):
                    final_result += f"{step_idx}. {step}\n"
                final_result += "\n"
        
        # Removed duplicate nutrition display - already shown in Nutrition tab
        
        if recipe.get('_warning'):
            final_result += f"⚠️ **{recipe['_warning']}**\n\n"
        
        final_result += f"📄 `{recipe.get('json_path', 'N/A')}`\n\n---\n\n"
    
    final_result += f"""
✅ All recipes generated, auto-fixed, and saved!

📁 **Location**: `{session_dir}`

🔧 **Improvements Applied**:
- ✅ Recipes include Prep Time, Cook Time, Total Time
- ✅ Smart servings extraction (from nutrition or recipe content)
- ✅ Corrected wrong titles
- ✅ Fixed ingredient formatting
- ✅ Detected truncated content
"""
    
    yield final_result

print("✓ Gradio functions defined (removed duplicate nutrition display)")

✓ Gradio functions defined (removed duplicate nutrition display)


## 13. Launch Gradio Web Interface

In [17]:
with gr.Blocks(title="cAIuldron - AI Recipe Generator", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # 🍳 cAIuldron - AI Recipe Generator
    
    Transform ingredient photos into delicious recipes!
    
    **100% Free • Runs Locally • Multi-Ingredient Detection • Auto-Fix**
    """)
    
    detection_data_state = gr.State(value="{}")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Upload Photo")
            image_input = gr.Image(type="pil", label="Ingredient Photo")
            
            gr.Markdown("### ⚙️ Detection Settings")
            confidence_slider = gr.Slider(
                minimum=0.05,
                maximum=0.50,
                value=0.15,
                step=0.01,
                label="Confidence Threshold"
            )
            
            detect_btn = gr.Button("🔍 Detect Ingredients", variant="primary", size="lg")
            
            gr.Markdown("### 🤖 Recipe Generation Model")
            model_selector = gr.Dropdown(
                choices=[
                    "GPT-2 (Fast)",
                    "Llama 3.2 1B (Recommended)",
                    "Llama 3.1 8B (Best Quality)"
                ],
                value="Llama 3.2 1B (Recommended)",
                label="Select Model"
            )
            
            generate_btn = gr.Button("🍳 Generate Recipes", variant="secondary", size="lg")
            
            gr.Markdown("""
            ### 💡 Usage:
            1. **Detect** (~1s) - Detect ingredients
            2. **Select Model** - Choose your model
            3. **Generate** (~30-60s) - Generate recipes with auto-fix
            
            ### 🤖 Model Comparison:
            - **GPT-2**: Fast, good quality
            - **Llama 1B**: Recommended, excellent quality ⭐
            - **Llama 8B**: Best quality, slower
            """)
        
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Results")
            
            with gr.Tabs():
                with gr.Tab("🔍 Detection"):
                    detection_output = gr.Markdown(value="Upload an image and click 'Detect Ingredients'")
                
                with gr.Tab("🥗 Nutrition"):
                    nutrition_output = gr.Markdown(value="Nutrition info will appear after detection")
                
                with gr.Tab("🍳 Recipes"):
                    recipes_output = gr.Markdown(value="Select a model and click 'Generate Recipes'")
    
    gr.Markdown("""
    ---
    **Powered by:** 
    - 🔍 CLIP + DETR (Ingredient Detection)
    - 🤖 GPT-2 / Llama 3.2 1B / Llama 3.1 8B (Recipe Generation)
    - 🥗 USDA FoodData Central (Nutrition Database)
    - 🔧 Auto-fix (Title & Format Correction)
    """)
    
    # Connect buttons
    detect_btn.click(
        fn=gradio_detect_ingredients,
        inputs=[image_input, confidence_slider],
        outputs=[detection_output, nutrition_output, detection_data_state]
    )
    
    generate_btn.click(
        fn=gradio_generate_recipes,
        inputs=[detection_data_state, model_selector],
        outputs=[recipes_output]
    )

print("\n" + "="*80)
print("🚀 LAUNCHING WEB INTERFACE")
print("="*80)
print("\nYour Trained Models:")
print("  ✅ GPT-2 Fine-tuned (Fast)")
print("  ✅ Llama 3.2 1B Fine-tuned with LoRA (Recommended)")
print("  📦 Llama 3.1 8B GGUF (Base model, not trained)")
print("\nFeatures:")
print("  🔍 Multi-ingredient detection (CLIP + DETR)")
print("  🤖 3 model options (2 are your trained models)")
print("  🥗 Nutrition estimation (529 ingredients)")
print("  🔧 Auto-fix titles & formatting")
print("\nInterface: http://127.0.0.1:7861")
print("To stop: Press Stop button or Kernel → Interrupt")
print("="*80 + "\n")

app.launch(
    inbrowser=True,
    server_port=7861,
    share=False
)


🚀 LAUNCHING WEB INTERFACE

Your Trained Models:
  ✅ GPT-2 Fine-tuned (Fast)
  ✅ Llama 3.2 1B Fine-tuned with LoRA (Recommended)
  📦 Llama 3.1 8B GGUF (Base model, not trained)

Features:
  🔍 Multi-ingredient detection (CLIP + DETR)
  🤖 3 model options (2 are your trained models)
  🥗 Nutrition estimation (529 ingredients)
  🔧 Auto-fix titles & formatting

Interface: http://127.0.0.1:7861
To stop: Press Stop button or Kernel → Interrupt

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Loading Llama 3.2 1B (Recommended)...
  ✓ Base model loaded (4-bit), Memory: 0.94 GB
  ✓ LoRA adapter merged - Using your fine-tuned model!
